# Buổi 3 — Python căn bản và TF

**Mục tiêu hôm nay:** đọc kiểu dữ liệu, viết preprocess, tạo vocabulary chung, đếm TF. Đây là ba viên gạch đầu của notebook kiểm tra đạo văn.

## Cách học notebook này
1. Chạy từng cell theo thứ tự.
2. Trước khi bấm Run, tự đoán output.
3. Tới cell có chữ TỰ LÀM, tự gõ trước.
4. Chỉ mở phần lời giải cuối notebook sau khi thử.

Không cần hiểu hết một lúc. Khi gặp lỗi, đọc dòng báo lỗi đầu tiên; lỗi thụt lề thường do các dòng trong hàm không cùng mức 4 dấu cách.

## 1. Chuẩn bị và kiểu dữ liệu

Notebook cần NumPy để tạo vector số. Trên Colab thường đã có sẵn; nếu thiếu, bỏ dấu # ở dòng pip bên dưới, chạy cell đó, rồi chạy lại cell import.

In [ ]:
# Nếu gặp ModuleNotFoundError, bỏ dấu # ở dòng dưới rồi chạy:
# %pip install numpy

import numpy as np

Một biến là tên mình đặt cho một giá trị. Ta sẽ gặp bốn kiểu chính:

- str: một đoạn chữ.
- list: nhiều giá trị theo thứ tự.
- set: nhiều giá trị nhưng không lặp.
- dict: tra một giá trị bằng khóa.

Ngoặc vuông dùng cho list và dấu hai chấm tách khóa với giá trị trong dict.

In [ ]:
sentence = "Mèo ăn cá"
words = ["mèo", "ăn", "cá", "mèo"]
unique_words = set(words)
counts = {"mèo": 2, "ăn": 1}

print(type(sentence).__name__)
print(words[0])
print(sorted(unique_words))
print(counts.get("cá", 0))

## 2. preprocess — đổi câu thành list từ

Hàm nhận một câu str, đổi về chữ thường, rồi tách ở khoảng trắng. return gửi kết quả cho nơi gọi; print chỉ hiện kết quả trên màn hình.

In [ ]:
def preprocess(text):
    text = text.lower()
    words = text.split()
    return words

print(preprocess("Mèo ĂN Cá"))

**Tự đoán rồi chạy:** preprocess("MÈO, ăn cá!") sẽ cho ra gì? split chỉ cắt khoảng trắng nên dấu phẩy vẫn nằm với từ đầu tiên. Notebook gốc dùng cách tách đơn giản này; buổi này ta giữ cùng quy tắc.

In [ ]:
# TỰ LÀM: gọi preprocess với một câu bạn tự nghĩ ra.
# Trước khi chạy, ghi output dự đoán ở đây.
my_sentence = "Chó thích chạy ngoài sân"
print(preprocess(my_sentence))

## 3. create_corpus — xử lý toàn bộ câu

Vocabulary là danh sách từ duy nhất dùng làm cột. Ta tạo nó từ tất cả tài liệu, rồi sắp xếp. Nếu mỗi câu tự tạo bộ cột riêng, vị trí của số sẽ không còn thống nhất.

In [ ]:
def create_corpus(docs):
    processed_docs = []
    vocabulary_set = set()

    for text in docs:
        words = preprocess(text)
        processed_docs.append(words)
        for word in words:
            vocabulary_set.add(word)

    vocabulary = sorted(vocabulary_set)
    return processed_docs, vocabulary

docs = ["mèo ăn cá mèo", "chó ăn cá"]
processed_docs, vocabulary = create_corpus(docs)

print(processed_docs)
print(vocabulary)

Đọc vòng lặp:
- for text in docs lấy lần lượt từng câu.
- preprocess biến câu thành list.
- append giữ list đó làm một tài liệu đã xử lý.
- Vòng trong add từ vào set; từ trùng không tạo bản mới.
- return trả hai kết quả; hai tên bên trái nhận chúng theo đúng thứ tự.

Trong ví dụ, vocabulary phải là ['ăn', 'cá', 'chó', 'mèo'].

In [ ]:
# TỰ LÀM: thêm một câu mới, ví dụ "mèo uống sữa".
# Chạy lại create_corpus và quan sát từ nào mới xuất hiện.
docs.append("mèo uống sữa")
processed_docs, vocabulary = create_corpus(docs)
print(vocabulary)

## 4. compute_tf — đếm tỷ lệ từ trong một tài liệu

TF của một từ = số lần từ ấy xuất hiện chia cho tổng số từ trong tài liệu. Từ không xuất hiện nhận 0. Ta tạo vector theo thứ tự vocabulary, chứ không theo thứ tự từ trong câu.

In [ ]:
def compute_tf(word_list, vocabulary):
    word_count = {}
    for word in word_list:
        word_count[word] = word_count.get(word, 0) + 1

    total_words = len(word_list)
    tf_vector = []
    for word in vocabulary:
        count = word_count.get(word, 0)
        if total_words == 0:
            tf_vector.append(0)
        else:
            tf_vector.append(count / total_words)

    return np.array(tf_vector)

docs = ["mèo ăn cá mèo", "chó ăn cá"]
processed_docs, vocabulary = create_corpus(docs)
tf_first_doc = compute_tf(processed_docs[0], vocabulary)
print(vocabulary)
print(tf_first_doc)

Với ['ăn', 'cá', 'chó', 'mèo'], vector tài liệu đầu là [0.25, 0.25, 0.0, 0.5].
- .get(word, 0) bắt đầu số đếm từ 0 nếu từ chưa xuất hiện.
- len(word_list) là mẫu số; từ lặp vẫn được tính trong tổng.
- Nhánh total_words == 0 tránh chia cho 0.
- np.array đổi list số thành mảng để buổi sau nhân với IDF.

### Tự luyện
1. Tính bằng tay TF(cá) trong tài liệu đầu.
2. Gọi compute_tf với list rỗng; phải trả vector 0 có cùng số cột.
3. Che lời giải, viết lại ba hàm trong notebook trắng với câu khác.

In [ ]:
# TỰ LÀM: hãy đổi đầu vào bên dưới rồi đoán trước vector.
new_words = preprocess("cá cá bơi")
print(compute_tf(new_words, ["bơi", "cá", "mèo"]))

## Lời giải tham khảo

Mở phần này sau khi đã tự làm. Hãy đọc từng dòng và tự giải thích vì sao cần nó. Nếu cần nhìn lời giải ngay, cũng không sao; sau đó đóng nó lại và viết lại lần nữa từ trí nhớ.

In [ ]:
# Ví dụ kiểm tra list rỗng và tỷ lệ TF:
empty_vector = compute_tf([], vocabulary)
print(empty_vector)
print(len(empty_vector) == len(vocabulary))
print(compute_tf(["cá", "cá", "bơi"], ["bơi", "cá", "mèo"]))

## Tự chấm buổi 3

Bạn đạt mục tiêu khi có thể tự viết preprocess, create_corpus và compute_tf; nói được mỗi hàm nhận gì/trả gì; giải thích TF(mèo) = 2/4 = 0.5; và lý giải vì sao vector có bốn số. Nếu chưa đạt, ôn đúng hàm đang vướng rồi thử corpus mới.

Buổi sau ta dùng chính các hàm này để hoàn thành notebook kiểm tra đạo văn.